# 다중 horizon 온도 예측 모델 (15/30/45/60/90초)

코드는 `Temp_Prediction_Model.py` 에 있고, 이 노트북은 그걸 불러와 실제로 돌려서 **결과가 설득력이 있는지** 확인한다.

## 두 가지 입력 설계를 나란히 비교한다

- **PAST_ONLY** — 현재 T·level·직전 60~120초 heat/exch/feed 가동률만 사용. 미래는 전혀 모른다고 가정 — **실전에 그대로 쓸 수 있는 진짜 예측기.**
- **WITH_FUTURE** — 여기에 t~t+h 구간의 미래 heat/exch/feed 가동률까지 안다고 가정 (`ARX_Temp_Model.py` 와 같은 전제). 실전 예측기로는 못 쓰지만 "액추에이터 계획을 안다면 얼마나 더 좋아지는가"의 상한선 역할.

## 타깃은 온도 수준이 아니라 변화량이다

`ΔT(t → t+h)` 를 맞힌다. 수준을 그대로 맞히면 "지금 값 유지"만으로도 R² 가 높게 나와 모델이 뭘 배웠는지 알 수 없다. 그래서 모든 지표를 **"변화 없음(ΔT=0)" 순박한 예측 대비 개선율**로 낸다.

## 세 가지 단위로 모델을 만든다

탱크별(8개) / 타입별(P=폴리올 5탱크 통합, I=이소시아네이트 3탱크 통합) / 전체(8탱크 통합). 통합 모델에는 탱크 원-핫 더미를 추가해 탱크별 기준선 차이를 흡수한다.

## 두 알고리즘을 비교한다

선형(Ridge) vs 비선형(HistGradientBoostingRegressor). 비선형이 이기는 정도가 곧 액추에이터-온도 관계의 비선형성 크기다.

In [1]:
import os
import sys

import pandas as pd

sys.path.insert(0, os.path.abspath("."))
import Temp_Prediction_Model as tpm

pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 200)

print("탱크:", list(tpm.TANKS.keys()))
print("horizon(초):", tpm.HORIZONS_SEC)
print("그룹:", list(tpm.GROUPS.keys()))

탱크: ['P1', 'P2', 'P3', 'P4', 'P5', 'I1', 'I2', 'I3']
horizon(초): [15, 30, 45, 60, 90]
그룹: ['P타입(폴리올, 5탱크통합)', 'I타입(이소시아네이트, 3탱크통합)', '전체(8탱크통합)']


## 1. 전체 실행 — 8탱크 개별 + 타입별 통합 + 전체 통합 × 5 horizon × 2 입력설계 × 2 알고리즘

In [2]:
result = tpm.main()

완료: P1


완료: P2


완료: P3


완료: P4


완료: P5


...

완료: I2


완료: I3


완료: P타입(폴리올, 5탱크통합)


완료: I타입(이소시아네이트, 3탱크통합)


완료: 전체(8탱크통합)

=== 온도 예측 모델 비교 (전체) ===
                 그룹  horizon초        입력설계      알고리즘  n_train  n_test  RMSE_변화없음기준  RMSE_모델  개선율%     R2  R2_변화없음기준
                 P1        15   PAST_ONLY 선형(Ridge)    33229   14250       2.4804   1.9610  20.9 0.3750     0.0000
                 P1        30   PAST_ONLY 선형(Ridge)    33229   14250       4.3790   3.1900  27.2 0.4693    -0.0000
                 P1        45   PAST_ONLY 선형(Ridge)    33229   14250       6.0944   4.1672  31.6 0.5325    -0.0000
                 P1        60   PAST_ONLY 선형(Ridge)    33229   14250       7.6147   4.8950  35.7 0.5868    -0.0000
                 P1        90   PAST_ONLY 선형(Ridge)    33229   14250      10.0931   5.6542  44.0 0.6862    -0.0000
                 P1        15 WITH_FUTURE 선형(Ridge)    33229   14250       2.4804   1.9144  22.8 0.4043     0.0000
                 P1        30 WITH_FUTURE 선형(Ridge)    33229   14250       4.3790   3.0333  30.7 0.5202    -0.0000
                 P1        45 WITH_FUTUR

## 2. 탱크별 결과 — PAST_ONLY (실전 예측기 관점)

`개선율%` 이 "변화 없음"이라는 순박한 예측보다 얼마나 더 정확한지다. 0% 근처면 모델이 액추에이터 정보로 아무것도 못 얻는다는 뜻.

In [3]:
tank_names = list(tpm.TANKS.keys())
past_only = result[(result["입력설계"] == "PAST_ONLY") & (result["그룹"].isin(tank_names))]
piv = past_only.pivot_table(index=["그룹", "알고리즘"], columns="horizon초", values="개선율%")
piv

horizon초        15    30    45    60    90
그룹 알고리즘                                   
I1 비선형(GBR)   25.3  35.0  33.3  34.1  39.9
   선형(Ridge)  18.3  17.0  14.9  15.6  19.2
I2 비선형(GBR)    6.2  11.1  16.3  20.2  24.4
   선형(Ridge)   4.6   8.2  11.5  14.2  17.4
I3 비선형(GBR)   40.8  54.7  56.3  56.1  59.5
   선형(Ridge)  34.5  37.3  34.1  33.4  34.9
P1 비선형(GBR)   31.4  40.5  44.8  49.0  56.7
   선형(Ridge)  20.9  27.2  31.6  35.7  44.0
P2 비선형(GBR)   21.1  42.9  45.1  42.5  41.9
   선형(Ridge)  22.3  33.6  33.0  32.2  33.6
P3 비선형(GBR)   15.5  24.1  30.5  35.0  40.1
   선형(Ridge)   9.7  15.7  19.2  22.4  27.4
P4 비선형(GBR)   31.4  51.6  61.1  66.1  69.5
   선형(Ridge)  24.7  44.2  52.3  55.4  54.5
P5 비선형(GBR)   54.1  60.9  64.0  65.6  67.4
   선형(Ridge)  44.7  50.3  53.1  54.9  58.5

## 3. PAST_ONLY vs WITH_FUTURE — 미래 액추에이터 정보의 가치

WITH_FUTURE 가 PAST_ONLY 보다 얼마나 더 좋아지는지 보면, "지금 이 순간의 명령"이 아니라 "앞으로 몇 초간 나갈 명령 전체"를 알아야 하는 정도를 가늠할 수 있다 (짧은 horizon 일수록 차이가 작아야 정상 — 이미 지금 켜진 게 그 구간 대부분을 차지하니까).

In [4]:
cmp = result[result["그룹"].isin(tank_names)].pivot_table(
    index=["그룹", "알고리즘", "horizon초"], columns="입력설계", values="개선율%"
).reset_index()
cmp["WITH_FUTURE_이득"] = cmp["WITH_FUTURE"] - cmp["PAST_ONLY"]
cmp.sort_values(["그룹", "알고리즘", "horizon초"])

입력설계,그룹,알고리즘,horizon초,PAST_ONLY,WITH_FUTURE,WITH_FUTURE_이득
0,I1,비선형(GBR),15,25.3,27.0,1.7
1,I1,비선형(GBR),30,35.0,41.2,6.2
2,I1,비선형(GBR),45,33.3,46.1,12.8
3,I1,비선형(GBR),60,34.1,48.3,14.2
4,I1,비선형(GBR),90,39.9,49.7,9.8
5,I1,선형(Ridge),15,18.3,20.4,2.1
6,I1,선형(Ridge),30,17.0,28.0,11.0
7,I1,선형(Ridge),45,14.9,33.3,18.4
8,I1,선형(Ridge),60,15.6,36.8,21.2
9,I1,선형(Ridge),90,19.2,39.1,19.9


## 4. 통합 단위 비교 — 탱크별 vs 타입별(P/I) vs 전체

탱크별 평균 개선율과 타입/전체 통합 모델의 개선율을 나란히 놓고, 통합이 정보를 더 주는지(표본이 늘어서) 아니면 오히려 흐리는지(탱크마다 다른 물성을 뭉개서) 확인한다.

In [5]:
per_tank_avg = (past_only.groupby(["알고리즘", "horizon초"])["개선율%"]
                .mean().reset_index().assign(그룹="탱크별_평균"))
group_rows = result[(result["입력설계"] == "PAST_ONLY") & (~result["그룹"].isin(tank_names))]
level_cmp = pd.concat([
    per_tank_avg[["그룹", "알고리즘", "horizon초", "개선율%"]],
    group_rows[["그룹", "알고리즘", "horizon초", "개선율%"]],
])
level_cmp.pivot_table(index=["알고리즘", "그룹"], columns="horizon초", values="개선율%")

horizon초                            15       30       45      60       90
알고리즘      그룹                                                             
비선형(GBR)  I타입(이소시아네이트, 3탱크통합)  25.5000  40.1000  43.5000  45.400  50.1000
          P타입(폴리올, 5탱크통합)      45.6000  55.0000  59.0000  60.500  63.0000
          전체(8탱크통합)            43.9000  53.9000  57.6000  60.100  62.5000
          탱크별_평균               28.2250  40.1000  43.9250  46.075  49.9250
선형(Ridge) I타입(이소시아네이트, 3탱크통합)  18.4000  23.1000  22.2000  22.400  24.2000
          P타입(폴리올, 5탱크통합)      21.1000  24.9000  27.9000  31.100  37.4000
          전체(8탱크통합)            18.7000  22.2000  24.7000  27.200  32.1000
          탱크별_평균               22.4625  29.1875  31.2125  32.975  36.1875

## 5. 그래프 — horizon 에 따른 개선율 추이 (탱크별, PAST_ONLY, 비선형)

In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
matplotlib.rcParams["font.family"] = "Malgun Gothic"
matplotlib.rcParams["axes.unicode_minus"] = False

fig, ax = plt.subplots(figsize=(9, 5))
sub = past_only[past_only["알고리즘"] == "비선형(GBR)"]
for tank, d in sub.groupby("그룹"):
    d = d.sort_values("horizon초")
    ax.plot(d["horizon초"], d["개선율%"], marker="o", label=tank)
ax.axhline(0, color="#888", linewidth=0.8)
ax.set_xlabel("horizon (초)")
ax.set_ylabel("개선율 % (vs 변화없음 기준)")
ax.set_title("탱크별 PAST_ONLY 비선형 모델 — horizon 에 따른 개선율")
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.3)
fig.tight_layout()
os.makedirs(tpm.OUT_DIR, exist_ok=True)
fig.savefig(os.path.join(tpm.OUT_DIR, "horizon별_개선율_탱크별.png"), dpi=120)
fig

<Figure size 900x500 with 1 Axes>

## 6. 결론 (실측값 기준)

**1) 15~90초 전 온도 예측은 8개 탱크 전부에서 "변화 없음" 순박한 예측보다 확실히 낫다.**
PAST_ONLY·비선형(GBR) 기준 개선율이 90초에서 24.4%(I2, 가장 약함) ~ 69.5%(P4, 가장 강함)다.
즉 실전에서 쓸 수 있는(미래 액추에이터를 몰라도 되는) 예측기가 이미 설득력이 있다.

**2) horizon 이 길어질수록 오히려 더 잘 맞는다 — 짧은 horizon 이 더 어렵다.**
탱크별 평균 개선율(PAST_ONLY·GBR): 15초 28.2% → 30초 40.1% → 45초 43.9% → 60초 46.1% → 90초 49.9%.
15초처럼 짧은 구간은 열관성 때문에 온도가 거의 안 움직여 `ΔT≈0` 자체가 이미 좋은 예측이라, 모델이 그 위에 더 보탤 정보가 적다.
사용자가 처음에 "1분은 좀 긴 것 같다"고 했던 직관과 반대로, 데이터 상으로는 **더 짧은 horizon 이 오히려 모델링하기 어렵다.**

**3) 비선형(GBR)이 선형(Ridge)을 항상, 그리고 horizon 이 길수록 더 크게 이긴다.**
평균 격차가 15초 +5.8%p → 90초 +13.7%p 로 벌어진다. heat/exchanger/feed 가 온도에 미치는 영향이 순수 선형이 아니라는 뜻 —
실전 배포용으로는 비선형 모델을 쓰는 게 맞다.

**4) 미래 액추에이터 계획을 안다고 가정하면(WITH_FUTURE) 더 좋아지지만, 그 이득은 horizon 이 길 때만 크다.**
평균 격차가 15초 +1.9%p → 90초 +10.0%p. 15초는 "지금 켜진 상태"가 그 구간 대부분을 차지해서 미래 정보의 값어치가 작고,
90초는 그사이 액추에이터가 몇 번이고 바뀔 수 있어 계획을 아는 쪽이 확실히 유리해진다.
→ **실전 배포는 PAST_ONLY 로, "액추에이터 스케줄을 앞서 계획할 수 있다면" 단기 최적화 가능성이 있다는 정도로 WITH_FUTURE 를 참고.**

**5) 탱크를 묶어서(통합) 학습하면 개별로 따로 학습하는 것보다 항상 더 잘 맞는다 — 물성 차이가 걱정한 것만큼 문제되지 않는다.**
8탱크 개별 모델의 평균 개선율(46.1%, horizon 60초)보다 8탱크를 전부 묶은 통합 모델(60.1%)이 14%p 더 높다.
P타입(폴리올 5탱크 통합, 60.5%)과 I타입(이소시아네이트 3탱크 통합, 45.4%)을 따로 봐도 각각의 개별 탱크 평균보다 낫다.
표본이 늘어나며 얻는 이득이 탱크 간 물성 차이로 인한 손실보다 크다는 뜻 — **탱크별로 8개를 따로 운영할 필요 없이, 최소한 타입별(P/I) 통합 모델, 혹은 전체 통합 모델 하나로 시작해도 된다.**

**6) I2 는 모든 조건에서 가장 예측하기 어려운 탱크다.**
90초·GBR·PAST_ONLY 기준 24.4%로 8개 탱크 중 최저(다음 최저는 I1 39.9%). `code_ongoing/relay_diagnosis.ipynb`, `Exchanger_시도_총정리.ipynb` 에서도 I2 가 반복적으로 효과크기가 작게 나온 탱크였다 — 태그명 불규칙(heat/feeding 은 "Tank 2 MDI", cool/exchanger 는 "Tank 2-2 MDI") 이나 센서 배치를 재점검할 가치가 있다.

**추천 알고리즘: HistGradientBoostingRegressor (비선형), PAST_ONLY 입력 설계.**
탱크별 대신 **타입별(P/I) 또는 전체 통합 모델**로 시작 — 표본이 늘어 성능이 오히려 좋아지고 유지보수할 모델 수도 줄어든다.
필요하면 WITH_FUTURE 버전을 "액추에이터 스케줄 최적화" 실험용으로 별도 활용.
